In [1]:
using TextAnalysis: serialize
using SQLite
using DataFrames
using WordTokenizers
using StatsBase
using Serialization
using MLDataUtils
using TextAnalysis
using ProgressMeter
using MLJ
using MLJText
using MLJBase
using Languages
using ThreadsX

CountTransformer = @load CountTransformer pkg=MLJText
MultinomialNBClassifier = @load MultinomialNBClassifier pkg=NaiveBayes

[ Info: For silent loading, specify `verbosity=0`. 


import MLJText ✔
import MLJNaiveBayesInterface ✔

[ Info: For silent loading, specify `verbosity=0`. 


MLJNaiveBayesInterface.MultinomialNBClassifier

In [2]:
function load_data()
    println("Fetching data from db...")
    db = SQLite.DB("data/panslop.db")
    spam_db = DBInterface.execute(db, "SELECT * FROM full_text ORDER BY RANDOM() LIMIT 2000") |> DataFrame
    ham_db = DBInterface.execute(db, "SELECT * FROM ham_full_text ORDER BY RANDOM() LIMIT 2000") |> DataFrame

    # ham_linux_docs = read("ham/LINUX_DOCS.md", String)
    # println("Loaded.")

    spam = spam_db.text
    ham = ham_db.text
    # push!(ham, ham_linux_docs)

    return spam, ham
end

load_data (generic function with 1 method)

In [26]:
spam, ham = load_data()
println("$(length(spam)) spam files")
println("$(length(ham)) ham files")

Fetching data from db...
2000 spam files
2000 ham files


In [ ]:
# split test and train set with Julia's cool new MLDataUtils
# refs:
# https://discourse.julialang.org/t/simple-tool-for-train-test-split/473/4
# https://github.com/JuliaML/MLDataUtils.jl
train_ham, test_ham = splitobs(ham; at=0.8)
train_spam, test_spam = splitobs(spam; at=0.8)

In [30]:
# prepare labels on the train set
train_labels = vcat(repeat(["ham"], length(train_ham)), repeat(["spam"], length(train_spam)))
test_labels = vcat(repeat(["ham"], length(test_ham)), repeat(["spam"], length(test_spam)))

800-element Vector{String}:
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 ⋮
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"

In [31]:
corpus_train = vcat(train_spam, train_ham)
corpus_test = vcat(test_spam, test_ham)

800-element Vector{String}:
 "# 🧪 NeuroForge Native App - Ma" ⋯ 2820 bytes ⋯ "ield and start chatting!** 🎉\n\n"
 "# Code of Conduct\n\n## Our pled" ⋯ 1316 bytes ⋯ "-covenant.org/), version 2.1.\n"
 "# EXHAUSTIVE_LEGACY_ANALYSIS.m" ⋯ 39734 bytes ⋯ "t unresolved contradictions.\n"
 "# Contributing to Canonry\n\nTha" ⋯ 2018 bytes ⋯ "int\n```\n\nAll three must pass.\n"
 "# OfficeCLI\n\n> **OfficeCLI is " ⋯ 40472 bytes ⋯ "I/main/install.ps1 | iex\n-->\n"
 "<p align=\"center\">\n  <img src=" ⋯ 66280 bytes ⋯ "sub>Happy coding 🚀</sub></p>\n"
 "# SIROCCO — Phase P (prefill)\n" ⋯ 10077 bytes ⋯ "idence in the bundle above._\n"
 "# Changelog\n\nAll notable chang" ⋯ 21409 bytes ⋯ " with a custom SVG component\n"
 "# Changelog\n\n## [v1.0.0-rc.1] " ⋯ 5519 bytes ⋯ "`company_id` on these tables.\n"
 "# SessionRuntime Split — surfa" ⋯ 27863 bytes ⋯ "erkat_rpc::session_runtime`.\n"
 "# Contributing to Equibles\n\nTh" ⋯ 7108 bytes ⋯ "yright in your contributions.\n"
 "# Changelog\n\nAll notable chang" ⋯ 

In [33]:
println("Tokenising...")
tokenised_train = ThreadsX.map(doc -> TextAnalysis.tokenize(Languages.English(), doc), corpus_train)
tokenised_test = ThreadsX.map(doc -> TextAnalysis.tokenize(Languages.English(), doc), corpus_test)

Tokenising...


800-element Vector{Vector{String}}:
 ["#", "🧪", "NeuroForge", "Native", "App", "-", "Manual", "Testing", "Instructions", "*"  …  "the", "input", "field", "and", "start", "chatting", "!", "*", "*", "🎉"]
 ["#", "Code", "of", "Conduct", "#", "#", "Our", "pledge", "We", "want"  …  "Contributor", "Covenant", "]", "(", "https://www.contributor-covenant.org/", ")", ",", "version", "2.1", "."]
 ["#", "EXHAUSTIVE", "_", "LEGACY", "_", "ANALYSIS.md", "—", "FrankenPandas", "Date", ":"  …  "actions", "(", "section", "28.4", ")", ",", "not", "unresolved", "contradictions", "."]
 ["#", "Contributing", "to", "Canonry", "Thanks", "for", "your", "interest", "in", "contributing"  …  "run", "lint", "`", "`", "`", "All", "three", "must", "pass", "."]
 ["#", "OfficeCLI", ">", "*", "*", "OfficeCLI", "is", "the", "world", "'"  …  "|", "bash", "install-command-windows", ":", "irm", "https://raw.githubusercontent.com/iOfficeAI/OfficeCLI/main/install.ps", "1", "|", "iex", ">"]
 ["<p", "align=", "\"", "center", 

In [39]:
println("Computing features...")
mach1 = machine(CountTransformer(), tokenised_train) |> MLJ.fit!

# matrix of counts
X = MLJ.transform(mach1, tokenised_train)
y = coerce(train_labels, OrderedFactor)

Computing features...


[ Info: Training machine(CountTransformer(max_doc_freq = 1.0, …), …).


3200-element CategoricalArrays.CategoricalArray{String,1,UInt32}:
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 "ham"
 ⋮
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"
 "spam"

In [40]:
classifier = MultinomialNBClassifier()

MultinomialNBClassifier(
  alpha = 1)

In [41]:
mach2 = machine(classifier, X, y)

untrained Machine; caches model-specific representations of data
  model: MultinomialNBClassifier(alpha = 1)
  args: 
    1:	Source @187 ⏎ AbstractMatrix{Count}
    2:	Source @837 ⏎ AbstractVector{OrderedFactor{2}}


In [ ]:
MLJ.fit!(mach2, rows=1:length(corpus_train))

[ Info: Training machine(MultinomialNBClassifier(alpha = 1), …).

[294948] signal 15: Terminated
in expression starting at In[44]:1
unknown function (ip: 0x7f40c3aa0872)
unknown function (ip: 0x7f40c3a948bb)
unknown function (ip: 0x7f40c3a94903)
poll at /usr/lib/libc.so.6 (unknown line)
_ZN3zmq15socket_poller_t4waitEP18zmq_poller_event_til at /home/mel/.julia/artifacts/a95373603fb869dcd86b9f7191e57a4918ba6d2c/lib/libzmq.so (unknown line)
_ZN3zmq15proxy_steerableEPNS_13socket_base_tES1_S1_S1_ at /home/mel/.julia/artifacts/a95373603fb869dcd86b9f7191e57a4918ba6d2c/lib/libzmq.so (unknown line)
zmq_proxy at /home/mel/.julia/packages/ZMQ/yNY0H/src/bindings.jl:359 [inlined]
heartbeat_thread at /home/mel/.julia/packages/IJulia/Vl5w1/src/heartbeat.jl:15
jfptr_heartbeat_thread_10076 at /home/mel/.julia/compiled/v1.11/IJulia/nfu7T_bm1sE.so (unknown line)
jlcapi_heartbeat_thread_9159 at /home/mel/.julia/compiled/v1.11/IJulia/nfu7T_bm1sE.so (unknown line)
unknown function (ip: 0x7f40c3a97fd8)
unkno